In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pickle
import pandas as pd
import seaborn as sns

from scipy.stats import wasserstein_distance

In [ ]:
# Load real test set used for simulations
df_ehrs = pd.read_pickle('data/preprocessed/baseline_synthetic_data_3_3std.pkl')
list_data = df_ehrs['los'].to_list()
print(f'{np.median(list_data):.2f}')

In [ ]:
# Print median LOS and LOS >4 hours
for real_exp_folder in ['real_normal', 'real_mci_arrival_a', 'real_mci_arrival_b', 'real_mci_arrival_c', 'real_mci_arrival_d', 'real_mci_resource_a', 'real_mci_resource_b', 'real_mci_resource_c', 'real_mci_resource_d', 'real_mci_workflow_a', 'real_mci_workflow_b', 'real_mci_workflow_c', 'real_mci_workflow_d']:
    val_list = []
    all_list = []
    mimic_id_dict = {}
    los4_list = []

    for i in range(1,1001):
        real_output_folder = f'output_{i}'

        df_syn1 = pd.read_csv(f'experiments/{real_exp_folder}/{real_output_folder}/synthetic_ehr.csv')

        for index, row in df_syn1.iterrows():
            mimic_idx = row['mimic_id']
            if mimic_idx not in mimic_id_dict.keys():
                mimic_id_dict[mimic_idx] = []

            mimic_id_dict[mimic_idx].append(row['ed_los'])

        list_syn1 = np.array(df_syn1['ed_los'].to_list()) / 60

        val_list.append(np.median(list_syn1))
        all_list.extend(list(list_syn1))

        los4_list.append(np.sum(list_syn1 > 4) / len(list_syn1))

    print(real_exp_folder, f'{np.median(val_list):.2f}', f'{np.percentile(val_list,25):.2f}', f'{np.percentile(val_list,75):.2f}', f'{np.median(los4_list):.2f}', f'{np.percentile(los4_list,25):.2f}', f'{np.percentile(los4_list,75):.2f}')

In [ ]:
# Print median LOS for overall patient population under baseline system conditions
fig, axes = plt.subplots(1, 1, figsize=(3,1.5))
fig.dpi = 600

sns.kdeplot(list_data, color='black', zorder=3)

val_list = []
all_list = []
mimic_id_dict = {}
wd_list = []
for i in range(1,1001):
    real_exp_folder = 'real_normal'
    real_output_folder = f'output_{i}'

    df_syn1 = pd.read_csv(f'experiments/{real_exp_folder}/{real_output_folder}/synthetic_ehr.csv')

    for index, row in df_syn1.iterrows():
        mimic_idx = row['mimic_id']
        if mimic_idx not in mimic_id_dict.keys():
            mimic_id_dict[mimic_idx] = []

        mimic_id_dict[mimic_idx].append(row['ed_los'])

    list_syn1 = np.array(df_syn1['ed_los'].to_list()) / 60
    val_list.append(np.median(list_syn1))
    all_list.extend(list(list_syn1))

    sns.kdeplot(list_syn1, color='orange', alpha=0.2)

    wd_list.append(wasserstein_distance(list_data, list_syn1))
print(f'{np.median(val_list):.2f}', f'{np.percentile(val_list,25):.2f}', f'{np.percentile(val_list,75):.2f}', f'{np.median(wd_list):.2f}', f'{np.percentile(wd_list,25):.2f}', f'{np.percentile(wd_list,75):.2f}', f'{np.median(all_list):.2f}')
plt.grid(linestyle=':', which='both')
plt.xlim(0,24)
plt.ylim(0,0.15)
plt.xlabel('Length of stay (hours)')
plt.show()

In [ ]:
# Print median LOS across acuity and disposition groups under baseline system conditions
all_wd = []
for feature_name in ['acuity', 'disposition']:
    for feature_val in sorted(set(df_ehrs[feature_name])):
        list_data = df_ehrs[df_ehrs[feature_name] == feature_val]['los'].to_list()
        print(feature_name, feature_val, f'{np.median(list_data):.2f}')

        fig, axes = plt.subplots(1, 1, figsize=(3,1.5))
        fig.dpi = 600

        sns.kdeplot(list_data, color='black', zorder=3)

        val_list = []
        all_list = []
        mimic_id_dict = {}
        wd_list = []
        for i in range(1,1001):
            real_exp_folder = 'real_normal'
            real_output_folder = f'output_{i}'

            df_syn1 = pd.read_csv(f'experiments/{real_exp_folder}/{real_output_folder}/synthetic_ehr.csv', dtype={'acuity': str})
            df_syn1 = df_syn1[df_syn1[feature_name] == feature_val]

            for index, row in df_syn1.iterrows():
                mimic_idx = row['mimic_id']
                if mimic_idx not in mimic_id_dict.keys():
                    mimic_id_dict[mimic_idx] = []

                mimic_id_dict[mimic_idx].append(row['ed_los'])

            list_syn1 = np.array(df_syn1['ed_los'].to_list()) / 60

            if len(list_syn1):
                val_list.append(np.median(list_syn1))
                all_list.extend(list(list_syn1))

                sns.kdeplot(list_syn1, color='orange', alpha=0.2)

                wd_list.append(wasserstein_distance(list_data, list_syn1))
        print(f'{np.median(val_list):.2f}', f'{np.percentile(val_list,25):.2f}', f'{np.percentile(val_list,75):.2f}', f'{np.median(wd_list):.2f}', f'{np.percentile(wd_list,25):.2f}', f'{np.percentile(wd_list,75):.2f}')
        plt.xlim(0,24)
        plt.show()

In [ ]:
# Get LOS per patient per simulation
patient_idx_dict = {}
all_acc = []
tn_list = []
fp_list = []
fn_list = []
tp_list = []

acuity_dict = {'1': {}, '2': {}, '3': {}, '4': {}, '5': {}}
disp_dict = {'HOME': {}, 'WARD': {}, 'ICU': {}}

df_test = pd.read_csv(f'data/preprocessed/test.csv')

for i in range(1,1001):
    real_exp_folder = 'real_normal'
    real_output_folder = f'output_{i}'
    df_syn1 = pd.read_csv(f'experiments/{real_exp_folder}/{real_output_folder}/synthetic_ehr.csv', dtype={'acuity': str})

    df_syn1['new_ed_los_hours'] = df_syn1['ed_los'] / 60
    df_syn1 = df_syn1.rename(columns={"mimic_id": "stay_id"})

    merged_df = df_test.merge(df_syn1, on='stay_id', how='left')
    df_sub1 = merged_df[merged_df['stay_id'].isin(df_syn1['stay_id'])].copy()
    df_sub1['target_los'] = (df_sub1['new_ed_los_hours'] > 4).astype(int)

    for _, row in df_sub1.iterrows():
        patient_idx = row['stay_id']
        if patient_idx not in patient_idx_dict.keys():
            patient_idx_dict[patient_idx] = []
        patient_idx_dict[patient_idx].append(row['new_ed_los_hours'])

        acuity_idx = row['acuity']
        disp_idx = row['disposition_y']

        if patient_idx not in acuity_dict[acuity_idx].keys():
            acuity_dict[acuity_idx][patient_idx] = []
        acuity_dict[acuity_idx][patient_idx].append(row['new_ed_los_hours'])

        if patient_idx not in disp_dict[disp_idx].keys():
            disp_dict[disp_idx][patient_idx] = []
        disp_dict[disp_idx][patient_idx].append(row['new_ed_los_hours'])

In [ ]:
def compute_interval_for_patient(sim_values, true_val):
    sim_values = np.asarray(sim_values, dtype=float)
    sim_values = np.round(sim_values, 2)

    lower = np.min(sim_values)
    upper = np.max(sim_values)
    width = upper - lower
    covered = int(lower <= true_val <= upper)

    return lower, upper, width, covered

true_idx_dict = {}
for _, row in df_test.iterrows():
    patient_idx = row['stay_id']
    true_idx_dict[patient_idx] = row['ed_los_hours']

In [ ]:
# Get number of times patients were sampled
len_list = [len(val) for val in patient_idx_dict.values()]
plt.hist(len_list, bins=20, range=(0,20))
plt.xlabel('Sample count')
plt.ylabel('Number of patients (test set)')
print(np.mean(len_list))

In [ ]:
# Get overall coverage and width
all_patient_width = []
all_patient_coverage = []
for key, val in patient_idx_dict.items():
    true_val = true_idx_dict[key]
    lower, upper, width, covered = compute_interval_for_patient(val, true_val)
    all_patient_width.append(width)
    all_patient_coverage.append(covered)
print(round(np.sum(all_patient_coverage)/len(all_patient_coverage),2))
print(round(np.median(all_patient_width),2), round(np.percentile(all_patient_width,25),2), round(np.percentile(all_patient_width,75),2))

In [ ]:
# Get coverage and width per acuity and disposition
for acuity_idx in acuity_dict.keys():
    all_patient_width = []
    all_patient_coverage = []
    for key, val in acuity_dict[acuity_idx].items():
        true_val = true_idx_dict[key]

        lower, upper, width, covered = compute_interval_for_patient(val, true_val)

        all_patient_width.append(width)
        all_patient_coverage.append(covered)

    print('---', acuity_idx)
    print(round(np.sum(all_patient_coverage)/len(all_patient_coverage),2))
    print(round(np.median(all_patient_width),2), round(np.percentile(all_patient_width,25),2), round(np.percentile(all_patient_width,75),2))

for disp_idx in disp_dict.keys():
    all_patient_width = []
    all_patient_coverage = []
    for key, val in disp_dict[disp_idx].items():
        true_val = true_idx_dict[key]

        lower, upper, width, covered = compute_interval_for_patient(val, true_val)

        all_patient_width.append(width)
        all_patient_coverage.append(covered)

    print('---', disp_idx)
    print(round(np.sum(all_patient_coverage)/len(all_patient_coverage),2))
    print(round(np.median(all_patient_width),2), round(np.percentile(all_patient_width,25),2), round(np.percentile(all_patient_width,75),2))